# NB_ingest_to_satellites — Silver → Satellites (append-only)

For each satellite defined in the model config:
1. Compute the hub hash key (HK) and diff hash (DIFF_HK) from tracked columns
2. LEFT JOIN against the latest DIFF_HK already stored in the satellite
3. INSERT only rows where DIFF_HK has changed (new row) or no previous row exists

Satellites are **append-only** — no updates or deletes, change detection via DIFF_HK.

In [ ]:
%run ./NB_dv_metadata

In [ ]:
dbutils.widgets.text("CATALOG", "workspace", "Unity Catalog name")
dbutils.widgets.text("VAULT_SCHEMA", "vault", "Vault schema")
dbutils.widgets.text("SILVER_SCHEMA", "silver", "Silver schema")
dbutils.widgets.text("MODEL_PATH", "pipeline_configs/datavault/dv_model.json", "DV Model JSON Path")
dbutils.widgets.text("WATERMARK_TS", "", "Optional: only process records >= this timestamp")
dbutils.widgets.text("FULL_RELOAD", "false", "Drop and recreate satellite tables before loading")

CATALOG       = dbutils.widgets.get("CATALOG")
VAULT_SCHEMA  = dbutils.widgets.get("VAULT_SCHEMA")
SILVER_SCHEMA = dbutils.widgets.get("SILVER_SCHEMA")
MODEL_PATH    = dbutils.widgets.get("MODEL_PATH")
WATERMARK_TS  = dbutils.widgets.get("WATERMARK_TS")
FULL_RELOAD   = dbutils.widgets.get("FULL_RELOAD").strip().lower() == "true"

model = load_model(MODEL_PATH)
print(f"Loaded {len(model['satellites'])} satellites from model")
if FULL_RELOAD:
    print("FULL_RELOAD=true: satellite tables will be dropped and recreated")

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

sat_new_rows = {}

for sat_cfg in model['satellites']:
    if not sat_cfg.get('enabled', True):
        print(f"  Skipping disabled satellite: {sat_cfg['name']}")
        continue

    sat_name      = sat_cfg['name']
    source_tbl    = sat_cfg['source_table']
    parent_hub    = sat_cfg['parent_hub']
    hk_src_col    = sat_cfg['hub_key_source_column']
    tracked_cols  = sat_cfg['tracked_columns']
    load_dt_col   = sat_cfg['load_date_column']
    rec_src       = sat_cfg['record_source']
    hk_col        = f"HK_{parent_hub[4:]}"
    target_tbl    = f"{CATALOG}.{VAULT_SCHEMA}.{sat_name.lower()}"

    if FULL_RELOAD:
        spark.sql(f"DROP TABLE IF EXISTS {target_tbl}")
        print(f"  Dropped {target_tbl}")

    # Ensure target table exists
    create_sat_table(sat_cfg)

    # Read Silver source
    src_df = spark.table(source_tbl)
    if WATERMARK_TS:
        wm_col = 'last_updated_dt' if 'last_updated_dt' in src_df.columns else hk_src_col
        src_df = src_df.filter(F.col(wm_col) >= WATERMARK_TS)

    # Compute hub hash key and diff hash
    src_df = (
        src_df
        .withColumn(hk_col, F.sha2(
            F.concat_ws('||', F.upper(F.trim(F.col(hk_src_col).cast('string')))), 256))
        .withColumn('DIFF_HK', generate_diff_hash(tracked_cols))
        .withColumn('LOAD_DATE', F.current_timestamp())
        .withColumn('RECORD_SOURCE', F.lit(rec_src))
        .select(hk_col, 'LOAD_DATE', 'DIFF_HK', *tracked_cols, 'RECORD_SOURCE')
    )

    # Get latest DIFF_HK already stored per hub key (for change detection)
    try:
        latest_hk_df = get_latest_diff_hash(target_tbl, hk_col)
        # LEFT JOIN: keep only rows where DIFF_HK changed or no prior row
        new_rows_df = (
            src_df.alias('src')
            .join(latest_hk_df.alias('lat'), on=hk_col, how='left')
            .filter(
                F.col('lat.DIFF_HK').isNull() |
                (F.col('src.DIFF_HK') != F.col('lat.DIFF_HK'))
            )
            .select('src.*')
        )
    except Exception:
        # Table is empty — insert all rows
        new_rows_df = src_df

    # Append new/changed rows only
    new_rows_df.write.format('delta').mode('append').saveAsTable(target_tbl)

    new_count = new_rows_df.count()
    sat_new_rows[sat_name] = new_count
    print(f"  {sat_name}: {new_count:,} new rows appended → {target_tbl}")

In [ ]:
print('\nSatellite ingestion complete.')
for sat_name, cnt in sat_new_rows.items():
    print(f'  {sat_name}: {cnt:,} new rows')